In [1]:
# imports

import pandas as pd
import numpy as np
import json
import polars as pl
from pathlib import Path


In [2]:
# Helpers to read in the json data and convert to dataframe

def load_json_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

def convert_to_dataframe(data, is_full_cleaned=False):
    record = []
    for table_id, details in data.items():
        statements = details[0]
        enatiled_statuses = details[1]
        caption = details[-1]
        
        if is_full_cleaned:
            pos_tags = details[2]
            for statement, entailed_status, statement_pos_tags in zip(statements, enatiled_statuses, pos_tags):
                record.append({
                    "table_id": table_id,
                    "statement": statement,
                    "entailed_status": entailed_status,
                    "caption": caption,
                    "pos_tags": statement_pos_tags
                }) 
        else:
            for statement, entailed_status in zip(statements, enatiled_statuses):
                record.append({
                    "table_id": table_id,
                    "statement": statement,
                    "entailed_status": entailed_status,
                    "caption": caption
                })
    return pd.DataFrame(record)

def load_and_convert_data_to_dataframe(path, is_full_cleaned=False):
    raw_data = load_json_data(path)
    
    return convert_to_dataframe(raw_data, is_full_cleaned=is_full_cleaned)

In [3]:
# Load in table dataframe

df_tables = (
    pl.read_csv(
        "../tabfact_data/tables/*.csv",
        has_header=False,
        separator="\1", # Non-existent delimiter so full line/file stays intact
        new_columns=["table_content"],
        include_file_paths="file_name",
    )
    .group_by("file_name", maintain_order=True) # Group all the table lines together 
    .agg(pl.col("table_content").str.join("\n")) # Concatenate table lines together
    .to_pandas()
)

df_tables["file_name"] = df_tables["file_name"].apply(lambda p: Path(p).name) # Only keep the table id

df_tables.shape

(16573, 2)

In [4]:
# Load in non table data frames

df_full_cleaned = load_and_convert_data_to_dataframe("../tabfact_data/full_cleaned.json", is_full_cleaned=True)

df_full_cleaned_refuted = df_full_cleaned[df_full_cleaned['entailed_status'] == 0]
df_full_cleaned_entailed = df_full_cleaned[df_full_cleaned['entailed_status'] == 1]

df_r1_training = load_and_convert_data_to_dataframe("../tabfact_data/r1_training_all.json")
df_r2_training = load_and_convert_data_to_dataframe("../tabfact_data/r2_training_all.json")

df_training = load_and_convert_data_to_dataframe("../tabfact_data/train_examples.json")
df_test = load_and_convert_data_to_dataframe("../tabfact_data/test_examples.json")
df_validation = load_and_convert_data_to_dataframe("../tabfact_data/val_examples.json")
